In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
    print(f'Changed CWD to: {os.getcwd()}')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gensim'], check=True)
print('Dependencies ready.')

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import os
import matplotlib.pyplot as plt
from gensim.models import Word2Vec

---
## What is Word2Vec?

A word embedding maps each word in a vocabulary to a dense vector of real numbers (e.g. 200 dimensions). The key insight of **Word2Vec** (Mikolov et al., 2013) is the **distributional hypothesis**: *words that appear in similar contexts tend to have similar meanings*. If "hepatitis" and "cirrhosis" both appear near words like "liver", "inflammation", "chronic", their vectors should end up close in the embedding space.

### Skip-gram vs CBOW

Word2Vec offers two training objectives:

| Model | Task | Strength |
|---|---|---|
| **Skip-gram** (`sg=1`) | Given a centre word, predict its context words | Better for rare words and large datasets |
| **CBOW** (`sg=0`) | Given context words, predict the centre word | Faster, better for frequent words |

We use **skip-gram** here because biomedical corpora contain many rare disease names, drug names, and gene symbols that skip-gram handles better.

### Why domain-specific embeddings?

Generic Word2Vec trained on Wikipedia or Google News learns that *"apple"* is near *"fruit"* and *"microsoft"*. But in biomedical text:
- *"tumor"* should be near *"malignancy"*, *"carcinoma"*, not near *"swelling"*
- *"aspirin"* should be near *"ibuprofen"* (both NSAIDs), not near *"headache"*
- Abbreviations like *DM* (diabetes mellitus), *COPD*, *ALD* are meaningless in generic corpora

By training on NCBI Disease abstracts and DDI drug sentences, the embeddings capture biomedical semantics. This will directly improve downstream NER and RE models.

---
## 1. Collect Training Text

In [ ]:
PARQUET_BASE = 'hf://datasets/ncbi_disease@refs/convert/parquet/ncbi_disease'

df_ncbi_train = pd.read_parquet(f'{PARQUET_BASE}/train/0000.parquet')
df_ncbi_val   = pd.read_parquet(f'{PARQUET_BASE}/validation/0000.parquet')
df_ncbi_test  = pd.read_parquet(f'{PARQUET_BASE}/test/0000.parquet')

# NCBI: tokens are already split — join them back to raw sentences
ncbi_sentences = []
for df in [df_ncbi_train, df_ncbi_val, df_ncbi_test]:
    for tokens in df['tokens']:
        ncbi_sentences.append(' '.join(list(tokens)))

print(f'NCBI sentences  : {len(ncbi_sentences)}')

# DDI: raw sentence strings
df_ddi_train = pd.read_csv('data/ddi_train.csv')
df_ddi_test  = pd.read_csv('data/ddi_test.csv')
ddi_sentences = (
    df_ddi_train['sentence'].dropna().tolist() +
    df_ddi_test['sentence'].dropna().tolist()
)
# Deduplicate DDI sentences (many pairs share the same sentence)
ddi_sentences = list(dict.fromkeys(ddi_sentences))

print(f'DDI sentences   : {len(ddi_sentences)}')

all_raw_sentences = ncbi_sentences + ddi_sentences
print(f'Total sentences : {len(all_raw_sentences)}')

---
## 2. Preprocessing

In [ ]:
# Tokens that are purely punctuation — keep hyphened words intact
_PUNCT_RE  = re.compile(r'^[^\w-]+$')          # only non-word, non-hyphen chars
_SPLIT_RE  = re.compile(r'[,\.;\(\)\[\]/\\]+') # split on these characters

# Common single-char abbreviations / symbols to keep
_KEEP_SINGLE = {'a', 'i'}

def preprocess_sentence(text):
    """Lowercase, split on punctuation (keep hyphens), filter noise tokens."""
    text = text.lower()
    # Split on spaces first, then split each chunk on punctuation
    raw_tokens = []
    for chunk in text.split():
        parts = _SPLIT_RE.split(chunk)
        raw_tokens.extend(parts)

    tokens = []
    for tok in raw_tokens:
        tok = tok.strip()
        if not tok:
            continue
        if _PUNCT_RE.match(tok):
            continue
        if len(tok) == 1 and tok not in _KEEP_SINGLE:
            continue
        tokens.append(tok)
    return tokens


# Preprocess all sentences
tokenized_sentences = [preprocess_sentence(s) for s in all_raw_sentences]
# Drop empty sentences
tokenized_sentences = [s for s in tokenized_sentences if len(s) > 0]

# Stats
all_tokens  = [t for s in tokenized_sentences for t in s]
vocab_raw   = set(all_tokens)
avg_len     = np.mean([len(s) for s in tokenized_sentences])

print(f'Tokenized sentences    : {len(tokenized_sentences)}')
print(f'Total tokens           : {len(all_tokens)}')
print(f'Unique tokens (raw)    : {len(vocab_raw)}')
print(f'Avg sentence length    : {avg_len:.1f} tokens')
print(f'\nSample processed sentence:')
print(tokenized_sentences[0])

---
## 3. Train Word2Vec

In [ ]:
model = Word2Vec(
    sentences   = tokenized_sentences,
    vector_size = 200,
    window      = 5,
    min_count   = 2,
    workers     = 4,
    sg          = 1,     # skip-gram
    epochs      = 10,
    seed        = 42
)

print(f'Vocabulary size (min_count≥2): {len(model.wv)}')
print(f'Vector dimensions            : {model.vector_size}')

---
## 4. Explore Embeddings

In [ ]:
query_words = ['diabetes', 'cancer', 'aspirin', 'hypertension', 'infection']

for word in query_words:
    if word in model.wv:
        similar = model.wv.most_similar(word, topn=10)
        print(f'\nTop 10 similar to "{word}":')
        for w, score in similar:
            print(f'  {w:<30} {score:.4f}')
    else:
        print(f'\n"{word}" not in vocabulary (min_count filtered)')

In [ ]:
pairs = [
    ('diabetes', 'hypertension'),
    ('diabetes', 'surgery'),
    ('aspirin', 'ibuprofen'),
    ('aspirin', 'diabetes'),
    ('cancer', 'tumor'),
    ('cancer', 'prescription'),
]

print('Cosine similarity between word pairs:\n')
print(f'  {"Pair":<40}  Similarity')
print('  ' + '-' * 52)
for w1, w2 in pairs:
    if w1 in model.wv and w2 in model.wv:
        sim = model.wv.similarity(w1, w2)
        print(f'  ("{w1}", "{w2}"){"":<{38 - len(w1) - len(w2)}}  {sim:.4f}')
    else:
        missing = [w for w in [w1, w2] if w not in model.wv]
        print(f'  ("{w1}", "{w2}")  — skipped, not in vocab: {missing}')

---
## 5. t-SNE Visualisation

t-SNE compresses the 200-dimensional word vectors into 2D for plotting. Words that are semantically similar in the original space should cluster together.

In [ ]:
from sklearn.manifold import TSNE

# 50 disease-related seed words
disease_seeds = [
    'diabetes', 'cancer', 'tumor', 'hypertension', 'infection',
    'hepatitis', 'leukemia', 'lymphoma', 'carcinoma', 'fibrosis',
    'melanoma', 'anemia', 'dystrophy', 'neuropathy', 'myopathy',
    'syndrome', 'disorder', 'disease', 'mutation', 'deficiency',
    'malignancy', 'neoplasm', 'adenoma', 'sarcoma', 'glioma',
    'obesity', 'atherosclerosis', 'thrombosis', 'ischemia', 'stroke',
    'epilepsy', 'schizophrenia', 'dementia', 'parkinson', 'alzheimer',
    'asthma', 'pneumonia', 'sepsis', 'arthritis', 'osteoporosis',
    'retinopathy', 'nephropathy', 'colitis', 'cirrhosis', 'pancreatitis',
    'meningitis', 'encephalopathy', 'myocarditis', 'vasculitis', 'dermatitis'
]

# 50 non-disease words
nondisease_seeds = [
    'gene', 'protein', 'receptor', 'pathway', 'expression',
    'treatment', 'therapy', 'surgery', 'diagnosis', 'prognosis',
    'aspirin', 'insulin', 'warfarin', 'metformin', 'doxorubicin',
    'patient', 'study', 'results', 'analysis', 'clinical',
    'chromosome', 'allele', 'polymorphism', 'sequencing', 'genome',
    'cell', 'tissue', 'blood', 'liver', 'kidney',
    'inhibitor', 'activator', 'antibody', 'enzyme', 'substrate',
    'hospital', 'trial', 'control', 'placebo', 'cohort',
    'age', 'male', 'female', 'children', 'adults',
    'binding', 'signaling', 'transcription', 'translation', 'replication'
]

# Keep only words present in model vocab
dis_words    = [w for w in disease_seeds    if w in model.wv][:50]
nondis_words = [w for w in nondisease_seeds if w in model.wv][:50]

all_words   = dis_words + nondis_words
all_vectors = np.array([model.wv[w] for w in all_words])
all_labels  = ['Disease'] * len(dis_words) + ['Non-disease'] * len(nondis_words)

print(f'Disease words in vocab    : {len(dis_words)}/{len(disease_seeds)}')
print(f'Non-disease words in vocab: {len(nondis_words)}/{len(nondisease_seeds)}')
print('Running t-SNE...')

tsne = TSNE(n_components=2, perplexity=20, random_state=42, n_iter=1000)
coords = tsne.fit_transform(all_vectors)

fig, ax = plt.subplots(figsize=(14, 10))
colours = {'Disease': '#e74c3c', 'Non-disease': '#3498db'}
split_at = len(dis_words)

for grp, start, end in [('Disease', 0, split_at), ('Non-disease', split_at, len(all_words))]:
    ax.scatter(
        coords[start:end, 0], coords[start:end, 1],
        c=colours[grp], label=grp, s=60, alpha=0.8
    )

for i, word in enumerate(all_words):
    ax.annotate(word, (coords[i, 0], coords[i, 1]),
                fontsize=7, alpha=0.85,
                xytext=(3, 3), textcoords='offset points')

ax.set_title('t-SNE of Biomedical Word2Vec Embeddings\n(disease vs non-disease words)', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
plt.tight_layout()
plt.savefig('results/tsne_word2vec.png', dpi=150)
plt.show()
print('Saved to results/tsne_word2vec.png')

---
## 6. Save Model

In [ ]:
os.makedirs('models', exist_ok=True)

model.save('models/word2vec_biomedical.model')
model.wv.save('models/word2vec_biomedical.kv')

print('Saved:')
print('  models/word2vec_biomedical.model  (full model, can continue training)')
print('  models/word2vec_biomedical.kv     (vectors only, smaller, faster to load)')

---
## 7. Build Embedding Matrix for NER

The downstream BiLSTM NER model needs:
1. A **word → integer index** mapping (`word2idx`)
2. An **embedding matrix** of shape `(vocab_size, 200)` where row `i` is the vector for the word at index `i`

For words seen during Word2Vec training we copy the trained vector. For unseen words (too rare, or from a different split) we initialise with a small uniform random vector — the model will fine-tune these during NER training.

In [ ]:
# Build vocabulary from NCBI training set only
# (test/val words are unknown at deployment time)
ner_vocab = set()
for tokens in df_ncbi_train['tokens']:
    for tok in list(tokens):
        ner_vocab.add(tok.lower())

print(f'Unique words in NCBI train: {len(ner_vocab)}')

# Special tokens first: index 0 = <PAD>, index 1 = <UNK>
special_tokens = ['<PAD>', '<UNK>']
sorted_vocab   = special_tokens + sorted(ner_vocab)
word2idx       = {w: i for i, w in enumerate(sorted_vocab)}
vocab_size     = len(word2idx)
embed_dim      = model.vector_size   # 200

print(f'Total vocab size (with specials): {vocab_size}')

# Initialise matrix — zero for <PAD>, random for everything else
rng = np.random.default_rng(42)
embedding_matrix = rng.uniform(-0.25, 0.25, (vocab_size, embed_dim)).astype(np.float32)
embedding_matrix[0] = 0.0   # <PAD> stays zero

trained_count = 0
random_count  = 0

for word, idx in word2idx.items():
    if word in ('<PAD>', '<UNK>'):
        continue
    if word in model.wv:
        embedding_matrix[idx] = model.wv[word]
        trained_count += 1
    else:
        random_count += 1

coverage = trained_count / (trained_count + random_count)
print(f'\nEmbedding coverage:')
print(f'  Words with trained vectors : {trained_count}')
print(f'  Words with random vectors  : {random_count}')
print(f'  Coverage                   : {coverage:.2%}')

# Save
np.save('models/embedding_matrix.npy', embedding_matrix)
with open('models/word2idx.json', 'w') as f:
    json.dump(word2idx, f)

print(f'\nSaved:')
print(f'  models/embedding_matrix.npy  shape={embedding_matrix.shape}')
print(f'  models/word2idx.json         {vocab_size} entries')